In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import hydra
from omegaconf import OmegaConf

from genpp import BASE_DIR
from genpp.configs import register_resolvers

In [3]:
register_resolvers()

In [4]:
with hydra.initialize_config_dir(version_base=None, config_dir=str(BASE_DIR / "configs")):
    cfg = hydra.compose(config_name="base_emos")
    OmegaConf.resolve(cfg)

In [66]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup("fit")

train_dataloader = datamodule.train_dataloader()

In [67]:
from tqdm import tqdm, trange

for _ in trange(1):
    for batch in tqdm(train_dataloader):
        x, y = batch

  0%|          | 0/1 [00:00<?, ?it/s]/Users/moritzfeik/Developer/GENPP/.pixi/envs/dev/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
100%|██████████| 1/1 [00:02<00:00,  2.24s/it]


In [68]:
model = hydra.utils.instantiate(cfg.model)

In [88]:
res = model(x)

In [ ]:
import torch

num_samples = 50
params = res[0]

In [112]:
from einops import repeat

In [115]:
params["mu"].shape

torch.Size([8, 37, 31])

In [ ]:
# Sampling from a normal is easy
quantiles = torch.linspace(0, 1, num_samples + 2)[1:-1]  # Exclude the endpoints
quantiles = repeat(
    quantiles,
    "n -> b n h w",
    b=params["mu"].shape[0],
    h=params["mu"].shape[1],
    w=params["mu"].shape[2],
)
dists = torch.distributions.Normal(
    loc=params["mu"].unsqueeze(1), scale=params["sigma"].unsqueeze(1)
)
samples = dists.icdf(quantiles)

In [ ]:
# Sampling from a truncated normal is hard